# Pulsefield Mel frontend metamer experiment

This notebook is the single executable entrypoint for the experiment. Edit only the **Input** cell, then run all cells. It directly optimizes waveform samples from noise against two log-Mel frontends and returns WAV paths ordered from lower to higher Mel similarity. Waveform-domain distances are report-only and never enter the loss.

Exact candidate geometry: 24 kHz, 128 Mel bins, 10 ms / 240-sample hop, and `n_fft=win_length=960` for an exact 40 ms Hann window.

In [1]:
# Input: change AUDIO_PATH (and optionally the segment/output settings), then Run All.
from pathlib import Path

_cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (_cwd, *_cwd.parents) if (p / 'pyproject.toml').is_file()), _cwd)
AUDIO_PATH = PROJECT_ROOT / 'dataset/0/2183073/audio.ogg'
OUTPUT_DIR = PROJECT_ROOT / 'artifacts/mel_metamer_notebook_4paths'
START_SECONDS = 11.0
DURATION_SECONDS = 6.0
SEEDS = (0, 1, 2)
LISTENING_SEED = 0
LISTENING_PATHS_PER_SETTING = 4
MAX_STEPS = 3000
MIN_STEPS = 100
LEARNING_RATE = 0.003
TARGET_NORMALIZED_RMSE = 0.10
NOISE_STD = 0.05
SAVE_EVERY = 250
LOG_EVERY = 100
DEVICE = 'auto'  # auto | cpu | mps | cuda
VERIFY_AGAINST_REPOSITORY_FRONTEND = True

In [2]:
from __future__ import annotations

import json
import math
import os
import tempfile
import time
from dataclasses import asdict, dataclass
from typing import Any, Sequence

os.environ.setdefault('MPLCONFIGDIR', str(Path(tempfile.gettempdir()) / 'pulsefield-matplotlib'))
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torch.nn.functional as F
from IPython.display import display
from nnAudio.features import MelSpectrogram
from pydub import AudioSegment

EXPERIMENT_VERSION = 'pulsefield_mel_metamer_notebook_v1'
LOG_MEL_FLOOR = 1e-5

@dataclass(frozen=True)
class MelConfig:
    name: str
    sample_rate: int
    mel_bins: int
    hop_ms: int
    n_fft: int
    win_length: int
    fmin: float
    fmax: float

    @property
    def hop_length(self) -> int:
        return int(self.sample_rate * self.hop_ms / 1000)

FRONTENDS = (
    MelConfig('existing', 16000, 80, 10, 400, 400, 20.0, 8000.0),
    MelConfig('candidate_24k_128mel_40ms', 24000, 128, 10, 960, 960, 20.0, 12000.0),
)

@dataclass(frozen=True)
class OptimizationConfig:
    seeds: tuple[int, ...]
    max_steps: int
    min_steps: int
    learning_rate: float
    target_normalized_rmse: float
    noise_std: float
    save_every: int
    log_every: int

OPTIMIZATION = OptimizationConfig(
    seeds=tuple(SEEDS), max_steps=MAX_STEPS, min_steps=MIN_STEPS,
    learning_rate=LEARNING_RATE, target_normalized_rmse=TARGET_NORMALIZED_RMSE,
    noise_std=NOISE_STD, save_every=SAVE_EVERY, log_every=LOG_EVERY,
)

In [3]:
# Exact self-contained frontend and audio loading semantics.
def resolve_device(requested: str) -> torch.device:
    if requested != 'auto':
        device = torch.device(requested)
        if device.type == 'cuda' and not torch.cuda.is_available():
            raise RuntimeError('CUDA was requested but is unavailable')
        if device.type == 'mps' and not torch.backends.mps.is_available():
            raise RuntimeError('MPS was requested but is unavailable')
        return device
    if torch.cuda.is_available():
        return torch.device('cuda')
    if torch.backends.mps.is_available():
        return torch.device('mps')
    return torch.device('cpu')

def load_segment(path: Path, config: MelConfig, start_seconds: float, duration_seconds: float) -> np.ndarray:
    path = Path(path)
    format_name = path.suffix[1:] if path.suffix else None
    with path.open('rb') as handle:
        audio = AudioSegment.from_file(handle, format=format_name)
    audio = audio.set_frame_rate(config.sample_rate).set_channels(1)
    samples = np.asarray(audio.get_array_of_samples(), dtype=np.float32)
    if samples.size:
        peak = float(np.max(np.abs(samples)))
        if peak > 0:
            samples /= peak
    start = round(start_seconds * config.sample_rate)
    count = round(duration_seconds * config.sample_rate)
    end = start + count
    if start < 0 or count <= 0 or end > samples.size:
        raise ValueError(f'invalid {start_seconds=}, {duration_seconds=}; source has {samples.size/config.sample_rate:.3f}s')
    return samples[start:end].astype(np.float32, copy=False)

def make_mel_layer(config: MelConfig, device: torch.device) -> MelSpectrogram:
    layer = MelSpectrogram(
        sr=config.sample_rate, n_fft=config.n_fft, win_length=config.win_length,
        n_mels=config.mel_bins, hop_length=config.hop_length, window='hann',
        center=False, power=2.0, fmin=config.fmin, fmax=config.fmax, norm=1,
        trainable_mel=False, trainable_STFT=False, verbose=False,
    ).to(device)
    layer.eval().requires_grad_(False)
    return layer

def log_mel(waveform: torch.Tensor, layer: MelSpectrogram, config: MelConfig) -> torch.Tensor:
    if waveform.ndim != 1:
        raise ValueError(f'expected mono [samples], got {tuple(waveform.shape)}')
    frame_count = math.ceil(waveform.numel() / config.hop_length) if waveform.numel() else 0
    if frame_count == 0:
        return waveform.new_empty((0, config.mel_bins))
    required = (frame_count - 1) * config.hop_length + config.n_fft
    if waveform.numel() < required:
        waveform = F.pad(waveform, (0, required - waveform.numel()))
    mel = layer(waveform.unsqueeze(0)).squeeze(0).transpose(0, 1)
    return mel.clamp_min(LOG_MEL_FLOOR).log()[:frame_count]

def canonical_target(waveform: np.ndarray, config: MelConfig) -> np.ndarray:
    layer = make_mel_layer(config, torch.device('cpu'))
    with torch.no_grad():
        return log_mel(torch.from_numpy(waveform), layer, config).cpu().numpy().astype(np.float32, copy=False)

def write_wav(path: Path, waveform: np.ndarray, sample_rate: int) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    sf.write(path, np.clip(waveform, -1.0, 1.0), sample_rate, subtype='PCM_24')

In [4]:
# Metrics and diagnostics. Only normalized log-Mel MSE is optimized.
def mel_metrics(reconstructed: np.ndarray, target: np.ndarray) -> dict[str, float]:
    residual = reconstructed.astype(np.float64) - target.astype(np.float64)
    mse = float(np.mean(residual ** 2))
    variance = max(float(np.var(target.astype(np.float64))), 1e-12)
    target_linear = np.exp(target.astype(np.float64))
    reconstructed_linear = np.exp(reconstructed.astype(np.float64))
    return {
        'log_mel_mse': mse,
        'log_mel_rmse': math.sqrt(mse),
        'log_mel_normalized_rmse': math.sqrt(mse / variance),
        'log_mel_mae': float(np.mean(np.abs(residual))),
        'log_mel_max_abs': float(np.max(np.abs(residual))),
        'linear_mel_spectral_convergence': float(
            np.linalg.norm(reconstructed_linear - target_linear) / max(float(np.linalg.norm(target_linear)), 1e-12)
        ),
    }

def waveform_metrics(reconstructed: np.ndarray, reference: np.ndarray) -> dict[str, float | None]:
    estimate = reconstructed.astype(np.float64)
    target = reference.astype(np.float64)
    correlation = None if min(float(np.std(estimate)), float(np.std(target))) <= 1e-12 else float(np.corrcoef(estimate, target)[0, 1])
    target_energy = float(np.dot(target, target))
    if target_energy <= 1e-12:
        si_sdr = None
    else:
        projected = float(np.dot(estimate, target) / target_energy) * target
        noise = estimate - projected
        si_sdr = float(10 * np.log10((np.dot(projected, projected) + 1e-12) / (np.dot(noise, noise) + 1e-12)))
    return {
        'waveform_rmse_report_only': float(np.sqrt(np.mean((estimate - target) ** 2))),
        'waveform_correlation_report_only': correlation,
        'si_sdr_db_report_only': si_sdr,
        'reconstruction_rms': float(np.sqrt(np.mean(estimate ** 2))),
        'reconstruction_peak': float(np.max(np.abs(estimate))),
        'reconstruction_clipped_fraction': float(np.mean(np.abs(estimate) >= 0.999)),
    }

def plot_diagnostics(path: Path, target: np.ndarray, reconstructed: np.ndarray, history: list[dict[str, float]], title: str) -> None:
    residual = reconstructed - target
    extent = (0.0, target.shape[0] * 0.01, 0.0, float(target.shape[1]))
    common_min, common_max = float(min(target.min(), reconstructed.min())), float(max(target.max(), reconstructed.max()))
    residual_limit = max(float(np.max(np.abs(residual))), 1e-8)
    fig, axes = plt.subplots(4, 1, figsize=(11, 12), constrained_layout=True)
    for axis, values, heading in ((axes[0], target, 'Target log-Mel'), (axes[1], reconstructed, 'Reconstructed log-Mel')):
        image = axis.imshow(values.T, origin='lower', aspect='auto', extent=extent, vmin=common_min, vmax=common_max, cmap='magma')
        axis.set_title(heading); fig.colorbar(image, ax=axis, label='ln power')
    image = axes[2].imshow(residual.T, origin='lower', aspect='auto', extent=extent, vmin=-residual_limit, vmax=residual_limit, cmap='coolwarm')
    axes[2].set_title('Log-Mel residual (reconstructed - target)'); fig.colorbar(image, ax=axes[2], label='ln power residual')
    axes[3].semilogy([row['step'] for row in history], [row['normalized_mel_mse'] for row in history])
    axes[3].set(title='Optimization loss', xlabel='Adam step', ylabel='normalized log-Mel MSE'); axes[3].grid(True, which='both', alpha=0.3)
    for axis in axes[:3]: axis.set(xlabel='time (s)', ylabel='Mel bin')
    fig.suptitle(title); fig.savefig(path, dpi=140); plt.close(fig)

In [5]:
def optimize_one_seed(
    frontend: MelConfig, reference: np.ndarray, target: np.ndarray, run_dir: Path,
    seed: int, optimization: OptimizationConfig, device: torch.device,
) -> tuple[dict[str, Any], list[dict[str, Any]]]:
    torch.manual_seed(seed)
    if device.type == 'cuda': torch.cuda.manual_seed_all(seed)
    layer = make_mel_layer(frontend, device)
    cpu_layer = make_mel_layer(frontend, torch.device('cpu'))
    target_device = torch.from_numpy(target).to(device)
    target_variance = target_device.var(unbiased=False).clamp_min(1e-12)
    waveform = torch.nn.Parameter(torch.randn(reference.size, device=device) * optimization.noise_std)
    optimizer = torch.optim.Adam([waveform], lr=optimization.learning_rate)
    run_dir.mkdir(parents=True, exist_ok=False)
    history: list[dict[str, float]] = []
    audio_rows: list[dict[str, Any]] = []
    saved_steps: set[int] = set()
    started = time.perf_counter()

    def save_checkpoint(step: int, filename: str, kind: str) -> dict[str, Any]:
        samples = waveform.detach().cpu().numpy().astype(np.float32, copy=False)
        path = run_dir / filename
        write_wav(path, samples, frontend.sample_rate)
        with torch.no_grad():
            observed = log_mel(torch.from_numpy(samples), cpu_layer, frontend).numpy()
        metrics = mel_metrics(observed, target)
        row = {
            'frontend': frontend.name, 'seed': seed, 'step': step, 'kind': kind,
            'normalized_log_mel_rmse': metrics['log_mel_normalized_rmse'],
            'audio_path': str(path.resolve()),
        }
        audio_rows.append(row); saved_steps.add(step)
        return row

    save_checkpoint(0, 'step_0000.wav', 'checkpoint')
    converged_on_device = False
    final_step = 0
    for step in range(optimization.max_steps + 1):
        optimizer.zero_grad(set_to_none=True)
        observed = log_mel(waveform, layer, frontend)
        loss = torch.mean((observed - target_device) ** 2) / target_variance
        if not bool(torch.isfinite(loss).item()): raise FloatingPointError(f'non-finite loss: {frontend.name=} {seed=} {step=}')
        normalized_rmse = float(torch.sqrt(loss.detach()).cpu())
        if step == 0 or step % optimization.log_every == 0 or step == optimization.max_steps:
            history.append({'step': step, 'normalized_mel_mse': float(loss.detach().cpu()), 'normalized_mel_rmse': normalized_rmse, 'elapsed_seconds': time.perf_counter() - started})
            print(f'[{frontend.name} seed={seed}] step={step:04d} normalized_rmse={normalized_rmse:.6f}')
        final_step = step
        converged_on_device = step >= optimization.min_steps and normalized_rmse <= optimization.target_normalized_rmse
        if converged_on_device or step == optimization.max_steps: break
        loss.backward()
        if waveform.grad is None or not bool(torch.isfinite(waveform.grad).all().item()): raise FloatingPointError('non-finite waveform gradient')
        optimizer.step()
        with torch.no_grad(): waveform.clamp_(-1.0, 1.0)
        completed_step = step + 1
        if completed_step % optimization.save_every == 0:
            save_checkpoint(completed_step, f'step_{completed_step:04d}.wav', 'checkpoint')

    final_samples = waveform.detach().cpu().numpy().astype(np.float32, copy=False)
    final_path = run_dir / 'final.wav'
    write_wav(final_path, final_samples, frontend.sample_rate)
    final_mel = canonical_target(final_samples, frontend)
    final_mel_metrics = mel_metrics(final_mel, target)
    final_row = {
        'frontend': frontend.name, 'seed': seed, 'step': final_step, 'kind': 'final',
        'normalized_log_mel_rmse': final_mel_metrics['log_mel_normalized_rmse'],
        'audio_path': str(final_path.resolve()),
    }
    audio_rows.append(final_row)
    canonical_converged = final_step >= optimization.min_steps and final_mel_metrics['log_mel_normalized_rmse'] <= optimization.target_normalized_rmse
    metrics = {
        **final_mel_metrics, **waveform_metrics(final_samples, reference),
        'frontend': frontend.name, 'seed': seed, 'steps_completed': final_step,
        'elapsed_seconds': time.perf_counter() - started, 'convergence_target': optimization.target_normalized_rmse,
        'converged': canonical_converged, 'optimizer_device_reached_target': converged_on_device,
        'optimization_target': 'normalized_log_mel_mse_only',
        'waveform_metrics_used_as_optimization_target': False, 'history': history,
    }
    (run_dir / 'metrics.json').write_text(json.dumps(metrics, indent=2, sort_keys=True) + '\n')
    plot_diagnostics(run_dir / 'diagnostics.png', target, final_mel, history, f'{frontend.name}, seed {seed}, step {final_step}; normalized RMSE={final_mel_metrics["log_mel_normalized_rmse"]:.6f}')
    return metrics, audio_rows

In [6]:
def run_mel_metamer_experiment(
    audio_path: Path, output_dir: Path, start_seconds: float, duration_seconds: float,
    optimization: OptimizationConfig = OPTIMIZATION, device_name: str = DEVICE,
    listening_seed: int = LISTENING_SEED, paths_per_setting: int = LISTENING_PATHS_PER_SETTING,
) -> tuple[pd.DataFrame, dict[str, Any]]:
    audio_path, output_dir = Path(audio_path).resolve(), Path(output_dir).resolve()
    if not audio_path.is_file(): raise FileNotFoundError(audio_path)
    if output_dir.exists(): raise FileExistsError(f'Output exists; choose a new OUTPUT_DIR: {output_dir}')
    if not optimization.seeds or len(set(optimization.seeds)) != len(optimization.seeds): raise ValueError('SEEDS must be non-empty and unique')
    if not 0 <= optimization.min_steps <= optimization.max_steps: raise ValueError('require 0 <= MIN_STEPS <= MAX_STEPS')
    if listening_seed not in optimization.seeds: raise ValueError('LISTENING_SEED must be present in SEEDS')
    if paths_per_setting != 4: raise ValueError('this listening contract requires exactly four paths per setting')
    output_dir.mkdir(parents=True, exist_ok=False)
    device = resolve_device(device_name)
    started = time.perf_counter()
    references = {f.name: load_segment(audio_path, f, start_seconds, duration_seconds) for f in FRONTENDS}
    source_frontend = max(FRONTENDS, key=lambda f: f.sample_rate)
    write_wav(output_dir / 'source.wav', references[source_frontend.name], source_frontend.sample_rate)
    summary: dict[str, Any] = {
        'experiment_version': EXPERIMENT_VERSION, 'source': {
            'path': str(audio_path), 'start_seconds': start_seconds, 'duration_seconds': duration_seconds,
            'listening_wav': str((output_dir / 'source.wav').resolve()),
        }, 'device': str(device), 'torch_version': torch.__version__,
        'optimization': asdict(optimization), 'frontends': {},
    }
    all_audio_rows: list[dict[str, Any]] = []
    for frontend in FRONTENDS:
        reference = references[frontend.name]
        target = canonical_target(reference, frontend)
        frontend_dir = output_dir / frontend.name
        frontend_dir.mkdir(parents=True, exist_ok=False)
        np.save(frontend_dir / 'target_mel.npy', target)
        config_payload = {**asdict(frontend), 'hop_length': frontend.hop_length, 'window': 'hann', 'center': False, 'power': 2.0, 'norm': 1, 'log': 'natural', 'log_mel_floor': LOG_MEL_FLOOR}
        (frontend_dir / 'config.json').write_text(json.dumps(config_payload, indent=2, sort_keys=True) + '\n')
        runs = []
        for seed in optimization.seeds:
            metrics, audio_rows = optimize_one_seed(frontend, reference, target, frontend_dir / f'seed_{seed}', seed, optimization, device)
            runs.append(metrics); all_audio_rows.extend(audio_rows)
        summary['frontends'][frontend.name] = {'config': config_payload, 'target_shape': list(target.shape), 'runs': runs}
    all_runs = [run for data in summary['frontends'].values() for run in data['runs']]
    all_converged = all(run['converged'] for run in all_runs)
    medians = {name: float(np.median([run['log_mel_normalized_rmse'] for run in data['runs']])) for name, data in summary['frontends'].items()}
    summary['convergence_comparison'] = {
        'target_normalized_rmse': optimization.target_normalized_rmse,
        'all_runs_reached_common_target': all_converged,
        'interpret_frontend_null_space': all_converged,
        'status': 'comparable' if all_converged else 'not_comparable_do_not_interpret',
        'median_normalized_rmse_by_frontend': medians,
    }
    summary['total_elapsed_seconds'] = time.perf_counter() - started
    all_progression = pd.DataFrame(all_audio_rows)
    listening_groups = []
    for frontend in FRONTENDS:
        group = all_progression[(all_progression['frontend'] == frontend.name) & (all_progression['seed'] == listening_seed)]
        group = group.sort_values('normalized_log_mel_rmse', ascending=False).reset_index(drop=True)
        if len(group) < paths_per_setting: raise RuntimeError(f'not enough checkpoints for {frontend.name}: {len(group)}')
        positions = np.rint(np.linspace(0, len(group) - 1, paths_per_setting)).astype(int)
        selected = group.iloc[positions].copy().reset_index(drop=True)
        selected['similarity_rank_low_to_high'] = np.arange(paths_per_setting)
        listening_groups.append(selected)
    progression = pd.concat(listening_groups, ignore_index=True)
    progression = progression[['frontend', 'seed', 'similarity_rank_low_to_high', 'step', 'kind', 'normalized_log_mel_rmse', 'audio_path']]
    assert len(progression) == len(FRONTENDS) * paths_per_setting
    assert progression.groupby('frontend').size().eq(paths_per_setting).all()
    progression.to_json(output_dir / 'audio_path_progression.json', orient='records', indent=2)
    summary['listening_manifest'] = {'seed': listening_seed, 'paths_per_setting': paths_per_setting, 'rows': len(progression), 'all_saved_audio_rows': len(all_progression)}
    summary['audio_path_progression_json'] = str((output_dir / 'audio_path_progression.json').resolve())
    (output_dir / 'summary.json').write_text(json.dumps(summary, indent=2, sort_keys=True) + '\n')
    return progression, summary

In [7]:
# In-notebook validation: exact geometry, finite gradients, and optional parity with repository production code.
assert [(f.sample_rate, f.mel_bins, f.hop_length, f.n_fft, f.win_length, f.fmax) for f in FRONTENDS] == [
    (16000, 80, 160, 400, 400, 8000.0),
    (24000, 128, 240, 960, 960, 12000.0),
]
for frontend in FRONTENDS:
    probe = torch.randn(frontend.sample_rate // 20, requires_grad=True) * 0.05
    probe.retain_grad()
    probe_mel = log_mel(probe, make_mel_layer(frontend, torch.device('cpu')), frontend)
    probe_mel.square().mean().backward()
    assert probe_mel.shape == (5, frontend.mel_bins)
    assert probe.grad is not None and torch.isfinite(probe.grad).all()

if VERIFY_AGAINST_REPOSITORY_FRONTEND:
    from pulsefield_model.features.mel_base import MelCacheConfig as RepositoryMelConfig
    from pulsefield_model.features.mel_base import compute_log_mel_10ms
    rng = np.random.default_rng(7)
    for frontend in FRONTENDS:
        waveform = rng.normal(0, 0.05, frontend.sample_rate // 8 + 17).astype(np.float32)
        repository_config = RepositoryMelConfig(
            sample_rate=frontend.sample_rate, mel_bins=frontend.mel_bins, hop_ms=frontend.hop_ms,
            n_fft=frontend.n_fft, win_length=frontend.win_length, fmin=frontend.fmin, fmax=frontend.fmax,
        )
        expected = compute_log_mel_10ms(waveform, sample_rate=frontend.sample_rate, config=repository_config)
        observed = canonical_target(waveform, frontend)
        assert np.allclose(observed, expected, rtol=1e-5, atol=1e-6), frontend.name
print('Notebook validation passed for both frontends.')

Notebook validation passed for both frontends.


In [8]:
# Entrypoint: exactly four paths per Mel setting, ordered from lower to higher similarity.
audio_paths, summary = run_mel_metamer_experiment(
    AUDIO_PATH, OUTPUT_DIR, START_SECONDS, DURATION_SECONDS, OPTIMIZATION, DEVICE,
    LISTENING_SEED, LISTENING_PATHS_PER_SETTING,
)
print(f"status={summary['convergence_comparison']['status']}  output={OUTPUT_DIR.resolve()}")
display(audio_paths)
audio_paths

[existing seed=0] step=0000 normalized_rmse=1.146001
[existing seed=0] step=0100 normalized_rmse=0.302277


[existing seed=0] step=0200 normalized_rmse=0.218547
[existing seed=0] step=0300 normalized_rmse=0.179208


[existing seed=0] step=0400 normalized_rmse=0.154989
[existing seed=0] step=0500 normalized_rmse=0.138843


[existing seed=0] step=0600 normalized_rmse=0.126195
[existing seed=0] step=0700 normalized_rmse=0.116525


[existing seed=0] step=0800 normalized_rmse=0.110499
[existing seed=0] step=0900 normalized_rmse=0.103358


...

[candidate_24k_128mel_40ms seed=2] step=1100 normalized_rmse=0.104675


[candidate_24k_128mel_40ms seed=2] step=1200 normalized_rmse=0.100938


status=comparable  output=/Users/l/projects/Pulsefield-model/artifacts/mel_metamer_notebook_4paths


,frontend,seed,similarity_rank_low_to_high,step,kind,normalized_log_mel_rmse,audio_path
0,existing,0,0,0,checkpoint,1.146001,/Users/l/projects/Pulsefield-model/artifacts/m...
1,existing,0,1,250,checkpoint,0.195683,/Users/l/projects/Pulsefield-model/artifacts/m...
2,existing,0,2,750,checkpoint,0.115137,/Users/l/projects/Pulsefield-model/artifacts/m...
3,existing,0,3,956,final,0.099818,/Users/l/projects/Pulsefield-model/artifacts/m...
4,candidate_24k_128mel_40ms,0,0,0,checkpoint,1.117262,/Users/l/projects/Pulsefield-model/artifacts/m...
5,candidate_24k_128mel_40ms,0,1,500,checkpoint,0.144643,/Users/l/projects/Pulsefield-model/artifacts/m...
6,candidate_24k_128mel_40ms,0,2,750,checkpoint,0.121192,/Users/l/projects/Pulsefield-model/artifacts/m...
7,candidate_24k_128mel_40ms,0,3,1173,final,0.099990,/Users/l/projects/Pulsefield-model/artifacts/m...


,frontend,seed,similarity_rank_low_to_high,step,kind,normalized_log_mel_rmse,audio_path
0,existing,0,0,0,checkpoint,1.146001,/Users/l/projects/Pulsefield-model/artifacts/m...
1,existing,0,1,250,checkpoint,0.195683,/Users/l/projects/Pulsefield-model/artifacts/m...
2,existing,0,2,750,checkpoint,0.115137,/Users/l/projects/Pulsefield-model/artifacts/m...
3,existing,0,3,956,final,0.099818,/Users/l/projects/Pulsefield-model/artifacts/m...
4,candidate_24k_128mel_40ms,0,0,0,checkpoint,1.117262,/Users/l/projects/Pulsefield-model/artifacts/m...
5,candidate_24k_128mel_40ms,0,1,500,checkpoint,0.144643,/Users/l/projects/Pulsefield-model/artifacts/m...
6,candidate_24k_128mel_40ms,0,2,750,checkpoint,0.121192,/Users/l/projects/Pulsefield-model/artifacts/m...
7,candidate_24k_128mel_40ms,0,3,1173,final,0.099990,/Users/l/projects/Pulsefield-model/artifacts/m...
